## __Tópicos avanzados en Inteligencia Artificial 1 - MIA__

__Profesor__: Anthony D. Cho

__Ayudante__: Luis Oliveros

**Asunto**: Pytorch. Uso de Module API
*****

## Librerias

In [1]:
import warnings
warnings.filterwarnings('ignore')

from time import time
from numpy import argmin
from pandas import read_excel, DataFrame
import matplotlib.pyplot as plt
%matplotlib inline

## Pre-processing
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

## Pytorch
import torch
from torch.utils.data import DataLoader, TensorDataset
from torch import nn, optim

print(torch.__version__)
print("CUDA is available? ->", torch.cuda.is_available())

# Make device agnostic code
device = "cuda" if torch.cuda.is_available() else "cpu"
device

2.3.1+cpu
CUDA is available? -> False


'cpu'

## Dataset

<center>
    <img src=https://blog.certifiedmtp.com/wp-content/uploads/2024/07/ASTM-C39-Mastering-Compressive-Strength-Tests-on-Concrete.jpg width=800>
</center>

El hormigón es el material más importante en ingeniería civil. La resistencia a la compresión del hormigón es una función altamente no lineal de la edad y los ingredientes.

El conjunto de datos contiene información de los ingredientes de las mezclas con 1030 muestras. La descripción es la siguiente:

__Descripcion__:
<center>

| **Name**             | **Data Type** | **Measurement**    | **Description** |
|----------------------|---------------|--------------------|-----------------|
| Cement | Quantitative  | kg in a m3 mixture | Input Variable  |
| Blast Furnace Slag | Quantitative | kg in a m3 mixture | Input Variable |
| Fly Ash | Quantitative | kg in a m3 mixture | Input Variable |
| Water | Quantitative | kg in a m3 mixture | Input Variable |
| Superplasticizer | Quantitative | kg in a m3 mixture | Input Variable |
| Coarse Aggregate | Quantitative | kg in a m3 mixture | Input Variable |
| Fine Aggregate | Quantitative | kg in a m3 mixture | Input Variable |
| Age | Quantitative | Day (1~365) | Input Variable |
| Concrete compressive strength | Quantitative | MPa | Output Variable |

</center>

**Objetivo**: Predecir Concrete compressive strength (Strength)

* La data y detalles está completamente disponible en [UCI Repository: Concrete Compressive Strength](https://archive.ics.uci.edu/dataset/165/concrete+compressive+strength)

#### Carga de datos

In [2]:
## Load data
data = read_excel('https://github.com/adoc-box/Datasets/raw/refs/heads/main/Concrete_Data.xls')
data.head(4)

,Cement (component 1)(kg in a m^3 mixture),Blast Furnace Slag (component 2)(kg in a m^3 mixture),Fly Ash (component 3)(kg in a m^3 mixture),Water (component 4)(kg in a m^3 mixture),Superplasticizer (component 5)(kg in a m^3 mixture),Coarse Aggregate (component 6)(kg in a m^3 mixture),Fine Aggregate (component 7)(kg in a m^3 mixture),Age (day),"Concrete compressive strength(MPa, megapascals)"
0,540.0,0.0,0.0,162.0,2.5,1040.0,676.0,28,79.986111
1,540.0,0.0,0.0,162.0,2.5,1055.0,676.0,28,61.887366
2,332.5,142.5,0.0,228.0,0.0,932.0,594.0,270,40.269535
3,332.5,142.5,0.0,228.0,0.0,932.0,594.0,365,41.052780


In [3]:
data.describe()

,Cement (component 1)(kg in a m^3 mixture),Blast Furnace Slag (component 2)(kg in a m^3 mixture),Fly Ash (component 3)(kg in a m^3 mixture),Water (component 4)(kg in a m^3 mixture),Superplasticizer (component 5)(kg in a m^3 mixture),Coarse Aggregate (component 6)(kg in a m^3 mixture),Fine Aggregate (component 7)(kg in a m^3 mixture),Age (day),"Concrete compressive strength(MPa, megapascals)"
count,1030.000000,1030.000000,1030.000000,1030.000000,1030.000000,1030.000000,1030.000000,1030.000000,1030.000000
mean,281.165631,73.895485,54.187136,181.566359,6.203112,972.918592,773.578883,45.662136,35.817836
std,104.507142,86.279104,63.996469,21.355567,5.973492,77.753818,80.175427,63.169912,16.705679
min,102.000000,0.000000,0.000000,121.750000,0.000000,801.000000,594.000000,1.000000,2.331808
25%,192.375000,0.000000,0.000000,164.900000,0.000000,932.000000,730.950000,7.000000,23.707115
50%,272.900000,22.000000,0.000000,185.000000,6.350000,968.000000,779.510000,28.000000,34.442774
75%,350.000000,142.950000,118.270000,192.000000,10.160000,1029.400000,824.000000,56.000000,46.136287
max,540.000000,359.400000,200.100000,247.000000,32.200000,1145.000000,992.600000,365.000000,82.599225


## Preprocesamiento de datos

In [4]:
## Predictors and target assignment
X = data.iloc[:, :-1]
y = data.iloc[:, [-1]]

## Partition sets
X_trainVal, X_test, y_trainVal, y_test = train_test_split(X, y, random_state=84)
X_train, X_val, y_train, y_val = train_test_split(X_trainVal, y_trainVal, random_state=84)

## scaling
scale = StandardScaler().fit(X_train)
X_train = scale.transform(X_train)
X_val = scale.transform(X_val)
X_test = scale.transform(X_test)
X_trainVal = scale.transform(X_trainVal)

## Display data shape
print('(train shape) X: {}, y: {}'.format(X_train.shape, y_train.shape))
print('(val shape) X: {}, y: {}'.format(X_val.shape, y_val.shape))
print('(test shape) X: {}, y: {}'.format(X_test.shape, y_test.shape))
print('(train-Val shape) X: {}, y: {}'.format(X_trainVal.shape, y_trainVal.shape))

(train shape) X: (579, 8), y: (579, 1)
(val shape) X: (193, 8), y: (193, 1)
(test shape) X: (258, 8), y: (258, 1)
(train-Val shape) X: (772, 8), y: (772, 1)


In [5]:
BATCH_SIZE=16

## Convert Array to tensor
X_train_tensor = torch.FloatTensor(X_train)
X_val_tensor = torch.FloatTensor(X_val)
X_test_tensor = torch.FloatTensor(X_test)
X_trainVal_tensor = torch.FloatTensor(X_trainVal)

y_train_tensor = torch.FloatTensor(y_train.values)
y_val_tensor = torch.FloatTensor(y_val.values)
y_test_tensor = torch.FloatTensor(y_test.values)
y_trainVal_tensor = torch.FloatTensor(y_trainVal.values)

## Crear un gestor de datos de pytorch
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
trainVal_dataset = TensorDataset(X_trainVal_tensor, y_trainVal_tensor)

train_dataloader = DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_dataloader = DataLoader(dataset=val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_dataloader = DataLoader(dataset=test_dataset, batch_size=BATCH_SIZE, shuffle=False)
trainVal_dataloader = DataLoader(dataset=trainVal_dataset, batch_size=BATCH_SIZE, shuffle=True)

### Diseño del modelo

In [ ]:
class NeuralNet(nn.Module):

    ## Definición del constructor
    def __init__(self, in_features):
      
      ## Heredar caracteristicas de nn.Module
      super(NeuralNet, self).__init__()

      ## Declaración de los operadores
      self.linear = nn.Linear(in_features=in_features, out_features=12)
      self.Act = nn.ReLU()
      self.output = nn.Linear(in_features=12, out_features=1)

    ## Definición del flujo de operaciones del modelo
    def forward(self, x):
       
       x = self.linear(x)
       x = self.Act(x)
       x = self.output(x)
       return x

In [ ]:
## Instancia del modelo
model = NeuralNet(in_features=X_train.shape[1])

## Compiler setting
loss_fn = nn.MSELoss()
optimizer = optim.Adam(params=model.parameters(), lr=0.001)

print(model)

In [ ]:
NUM_EPOCHS = 300

## Performance allocation
performance = {'loss': [], 'val_loss': [], 
               'mae': [], 'val_mae': []}

start = time()
for epoch in range(NUM_EPOCHS):
    
    ## Set model to training mode
    model.train()

    ## cumulated loss
    loss_train_cum, loss_val_cum = 0, 0
    y_pred, y_true = [], []

    for (x_batch, y_batch) in train_dataloader:

        ## Compute prediction from model
        y_batch_pred = model(x_batch)

        ## Compute loss 
        loss = loss_fn(y_batch_pred, y_batch)

        ## Clear gradient, apply propagation, and model weight updating
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        ## Store cumulated loss
        loss_train_cum += loss.item() * x_batch.size(0)

        ## Store predicted and ground truth labels
        y_pred.extend(y_batch_pred.flatten().round().detach().numpy())
        y_true.extend(y_batch.flatten().detach().numpy())
    
    ## Compute epoch loss and metric
    epoch_train_loss = loss_train_cum / len(train_dataloader.dataset)
    epoch_train_mae = mean_absolute_error(y_pred=y_pred, y_true=y_true)
    performance['loss'].append(epoch_train_loss)
    performance['mae'].append( epoch_train_mae )

    ## Set model to evaluate mode
    model.eval()

    with torch.no_grad():

        y_pred, y_true = [], []
        for (x_batch, y_batch) in val_dataloader:

            ## Compute prediction from model
            y_batch_pred = model(x_batch)

            ## Compute loss 
            loss_val = loss_fn(y_batch_pred, y_batch)

            ## Store cumulated loss
            loss_val_cum += loss_val.item() * x_batch.size(0)

            ## Store predicted and ground truth labels
            y_pred.extend(y_batch_pred.flatten().round().numpy())
            y_true.extend(y_batch.flatten().numpy())

    ## Compute epoch loss and metric
    epoch_val_loss = loss_val_cum / len(val_dataloader.dataset)
    epoch_val_mae = mean_absolute_error(y_pred=y_pred, y_true=y_true)
    performance['val_loss'].append(epoch_val_loss)
    performance['val_mae'].append( epoch_val_mae )

    print('Epoch {:4}/{}, loss: {:.4f}, mae: {:.4f}, val_loss: {:.4f}, val_mae: {:.4f}'.format(
                                                                epoch+1, NUM_EPOCHS,
                                                                epoch_train_loss, epoch_train_mae,
                                                                epoch_val_loss, epoch_val_mae))

    ## Clear gradient
    optimizer.zero_grad()

stop = time()
print('Time spent[s]: {:2f}'.format(stop -start))

In [ ]:
performance = DataFrame(performance)#.plot(figsize=(15, 4))

performance[['loss', 'val_loss']].plot(figsize=(15, 4))
performance[['mae', 'val_mae']].plot(figsize=(15, 4))

In [ ]:
## Searching for the best epoch
id_min = argmin(performance['val_loss'])
print('Loss - Validation: {} - Error: {}'.format(id_min+1, performance['val_loss'][id_min]))

## Searching for the best epoch
id_min = argmin(performance['val_mae'])
print('MAE - Validation: {} - Error: {}'.format(id_min+1, performance['val_mae'][id_min]))

## Mejor modelo

In [ ]:
## Instancia del modelo
model = NeuralNet(in_features=X_train.shape[1])

## Compiler setting
loss_fn = nn.MSELoss()
optimizer = optim.Adam(params=model.parameters(), lr=0.001)

In [ ]:
NUM_EPOCHS = 297

start = time()
for epoch in range(NUM_EPOCHS):
    
    ## Set model to training mode
    model.train()

    ## cumulated loss
    loss_train_cum, loss_val_cum = 0, 0
    y_pred, y_true = [], []

    for (x_batch, y_batch) in trainVal_dataloader:

        ## Compute prediction from model
        y_batch_pred = model(x_batch)

        ## Compute loss 
        loss = loss_fn(y_batch_pred, y_batch)

        ## Clear gradient, apply propagation, and model weight updating
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        ## Store cumulated loss
        loss_train_cum += loss.item() * x_batch.size(0)

        ## Store predicted and ground truth labels
        y_pred.extend(y_batch_pred.flatten().round().detach().numpy())
        y_true.extend(y_batch.flatten().detach().numpy())
    
    ## Compute epoch loss and metric
    epoch_train_loss = loss_train_cum / len(train_dataloader.dataset)
    epoch_train_mae = mean_absolute_error(y_pred=y_pred, y_true=y_true)

    print('Epoch {:4}/{}, loss: {:.4f}, mae: {:.4f}'.format(
                                                                epoch+1, NUM_EPOCHS,
                                                                epoch_train_loss, epoch_train_mae,
                                                                ))

stop = time()
print('Time spent[s]: {:2f}'.format(stop -start))

In [ ]:
## Set model to evaluate mode
model.eval()

## Compute prediction
with torch.no_grad():
    prediction = model(X_test_tensor)

## Convert tensor to array
prediction = prediction.flatten().detach().numpy()
prediction

In [ ]:
## Display results
plt.figure(figsize=(18, 5))
plt.plot(y_test.values, label='Ground Truth')
plt.plot(prediction, label='Prediction')
plt.ylabel(data.columns[-1])
plt.xlabel('sample')
plt.title('Ground truth vs prediction')
plt.legend()
plt.tight_layout()
plt.show()